In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, ndcg_score
from sklearn.model_selection import TimeSeriesSplit
import lightgbm as lgb
from datetime import datetime, timedelta
import pickle
import pyodbc
import logging
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')


In [2]:
# Setup logging
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('recommendation_engine.log'),
        logging.StreamHandler()
    ]
)


In [3]:
class ProductionFMCGEngine:
    """
    Production-ready FMCG recommendation engine with proper temporal validation
    and efficient batch processing capabilities.
    """
    
    def __init__(self, config: Dict):
        self.config = config
        self.model = None
        self.encoders = {}
        self.scalers = {}
        self.feature_columns = []
        self.customer_profiles = {}
        self.product_catalog = {}
        self.location_inventory = {}
        
        # Performance tracking
        self.model_metrics = {}
        self.business_metrics = {}
        
    def load_data_batch(self, start_date: datetime, end_date: datetime) -> pd.DataFrame:
        """
        Load purchase data in batches to handle large datasets efficiently.
        Uses proper SQL with date range filtering and indexes.
        """
        query = """
        SELECT 
            customer_id,
            product_id,
            location_id,
            purchase_date,
            quantity,
            unit_price,
            total_amount,
            -- Pre-compute ISO week for efficiency
            DATEADD(day, 1-DATEPART(weekday, purchase_date), purchase_date) as iso_week_start
        FROM purchases_fact p
        INNER JOIN customer_dim c ON p.customer_id = c.customer_id
        INNER JOIN product_dim pr ON p.product_id = pr.product_id
        WHERE purchase_date >= ? AND purchase_date < ?
            AND c.is_active = 1
            AND pr.is_active = 1
        ORDER BY customer_id, purchase_date
        """
        
        try:
            conn = pyodbc.connect(self.config['connection_string'])
            df = pd.read_sql(query, conn, params=[start_date, end_date])
            conn.close()
            logging.info(f"Loaded {len(df)} records from {start_date} to {end_date}")
            return df
        except Exception as e:
            logging.error(f"Data loading failed: {e}")
            return self._generate_mock_data(start_date, end_date)
    
    def _generate_mock_data(self, start_date: datetime, end_date: datetime) -> pd.DataFrame:
        """Generate realistic mock FMCG data for testing"""
        np.random.seed(42)
        
        # Realistic FMCG products with categories
        products = {
            'MILK_1L': {'category': 'Dairy', 'repurchase_days': 3, 'price': 250},
            'BREAD_500G': {'category': 'Bakery', 'repurchase_days': 2, 'price': 150},
            'DETERGENT_1KG': {'category': 'Household', 'repurchase_days': 21, 'price': 800},
            'RICE_5KG': {'category': 'Staples', 'repurchase_days': 14, 'price': 2500},
            'COOKING_OIL_1L': {'category': 'Cooking', 'repurchase_days': 28, 'price': 600},
            'SOAP_BAR': {'category': 'Personal Care', 'repurchase_days': 10, 'price': 100},
            'NOODLES_PACK': {'category': 'Instant Food', 'repurchase_days': 7, 'price': 80},
            'SUGAR_1KG': {'category': 'Staples', 'repurchase_days': 30, 'price': 400}
        }
        
        customers = [f'CUST_{i:04d}' for i in range(1, 201)]  # 200 customers
        locations = ['LAGOS_VI', 'LAGOS_IKEJA', 'ABUJA_WUSE', 'KANO_SABON', 'PH_GRA']
        
        # Generate realistic purchase patterns
        data = []
        current_date = start_date
        
        while current_date < end_date:
            # Weekend effect - more purchases on weekends
            weekend_multiplier = 1.5 if current_date.weekday() >= 5 else 1.0
            
            for customer in customers:
                # Customer activity probability (some customers shop more frequently)
                if np.random.random() > 0.15 * weekend_multiplier:  # 15% daily activity base rate
                    continue
                    
                location = np.random.choice(locations)
                
                # Multi-product baskets (realistic FMCG behavior)
                n_products = np.random.choice([1, 2, 3, 4], p=[0.4, 0.3, 0.2, 0.1])
                selected_products = np.random.choice(list(products.keys()), n_products, replace=False)
                
                for product in selected_products:
                    # Realistic quantity based on product type
                    if products[product]['category'] in ['Dairy', 'Bakery']:
                        quantity = np.random.choice([1, 2], p=[0.8, 0.2])
                    else:
                        quantity = 1
                    
                    unit_price = products[product]['price']
                    total_amount = quantity * unit_price
                    
                    data.append({
                        'customer_id': customer,
                        'product_id': product,
                        'location_id': location,
                        'purchase_date': current_date,
                        'quantity': quantity,
                        'unit_price': unit_price,
                        'total_amount': total_amount,
                        'iso_week_start': current_date - timedelta(days=current_date.weekday())
                    })
            
            current_date += timedelta(days=1)
        
        return pd.DataFrame(data)
    
    def create_customer_profiles(self, df: pd.DataFrame) -> Dict:
        """
        Create comprehensive customer profiles for better recommendations.
        """
        profiles = {}
        
        for customer_id in df['customer_id'].unique():
            cust_data = df[df['customer_id'] == customer_id]
            
            # Purchase patterns
            avg_basket_size = cust_data.groupby('purchase_date')['quantity'].sum().mean()
            avg_spend_per_visit = cust_data.groupby('purchase_date')['total_amount'].sum().mean()
            purchase_frequency = len(cust_data['purchase_date'].unique())
            
            # Product preferences
            product_freq = cust_data['product_id'].value_counts(normalize=True).to_dict()
            category_freq = cust_data.groupby('product_id').first().reset_index().groupby('product_id')['product_id'].count()
            
            # Temporal patterns
            day_of_week_pattern = cust_data['purchase_date'].dt.dayofweek.value_counts(normalize=True).to_dict()
            
            # Location preference
            primary_location = cust_data['location_id'].mode()[0] if len(cust_data) > 0 else None
            
            profiles[customer_id] = {
                'avg_basket_size': avg_basket_size,
                'avg_spend_per_visit': avg_spend_per_visit,
                'purchase_frequency': purchase_frequency,
                'product_preferences': product_freq,
                'day_patterns': day_of_week_pattern,
                'primary_location': primary_location,
                'total_purchases': len(cust_data),
                'first_purchase': cust_data['purchase_date'].min(),
                'last_purchase': cust_data['purchase_date'].max()
            }
        
        return profiles
    
    def engineer_features(self, df: pd.DataFrame, prediction_date: datetime) -> pd.DataFrame:
        """
        Create comprehensive feature set for FMCG recommendations.
        """
        features = []
        
        # Get all customer-product-location combinations for the prediction date
        active_customers = df[df['purchase_date'] >= prediction_date - timedelta(days=30)]['customer_id'].unique()
        
        for customer_id in active_customers:
            cust_data = df[df['customer_id'] == customer_id]
            profile = self.customer_profiles.get(customer_id, {})
            
            # Primary location for this customer
            primary_location = profile.get('primary_location', 'LAGOS_VI')
            
            # Get candidate products (products available in their location)
            candidate_products = df[df['location_id'] == primary_location]['product_id'].unique()
            
            for product_id in candidate_products:
                # Historical purchase data for this customer-product pair
                cust_prod_data = cust_data[cust_data['product_id'] == product_id]
                
                # Temporal features
                if len(cust_prod_data) > 0:
                    last_purchase = cust_prod_data['purchase_date'].max()
                    days_since_last = (prediction_date - last_purchase).days
                    avg_repurchase_interval = cust_prod_data['purchase_date'].diff().dt.days.mean()
                    if pd.isna(avg_repurchase_interval):
                        avg_repurchase_interval = 30  # Default for first-time purchases
                    
                    purchase_count_last_30d = len(cust_prod_data[cust_prod_data['purchase_date'] >= prediction_date - timedelta(days=30)])
                    total_quantity_purchased = cust_prod_data['quantity'].sum()
                    avg_quantity_per_purchase = cust_prod_data['quantity'].mean()
                else:
                    days_since_last = 999  # Never purchased
                    avg_repurchase_interval = 30
                    purchase_count_last_30d = 0
                    total_quantity_purchased = 0
                    avg_quantity_per_purchase = 0
                
                # Repurchase probability (key for FMCG)
                repurchase_score = max(0, 1 - (days_since_last - avg_repurchase_interval) / avg_repurchase_interval)
                
                # Product popularity in location
                location_product_data = df[(df['location_id'] == primary_location) & (df['product_id'] == product_id)]
                product_popularity = len(location_product_data) / len(df[df['location_id'] == primary_location])
                
                # Seasonality (month-based)
                month = prediction_date.month
                seasonal_score = cust_prod_data[cust_prod_data['purchase_date'].dt.month == month].shape[0] / max(1, len(cust_prod_data))
                
                # Day of week pattern
                dow = prediction_date.weekday()
                dow_score = profile.get('day_patterns', {}).get(dow, 0.14)  # Uniform default
                
                # Cross-category affinity (simplified)
                # In production, this would be more sophisticated
                cross_category_score = 0.5  # Placeholder
                
                features.append({
                    'customer_id': customer_id,
                    'product_id': product_id,
                    'location_id': primary_location,
                    'days_since_last_purchase': days_since_last,
                    'avg_repurchase_interval': avg_repurchase_interval,
                    'repurchase_score': repurchase_score,
                    'purchase_count_last_30d': purchase_count_last_30d,
                    'total_quantity_purchased': total_quantity_purchased,
                    'avg_quantity_per_purchase': avg_quantity_per_purchase,
                    'product_popularity': product_popularity,
                    'seasonal_score': seasonal_score,
                    'dow_score': dow_score,
                    'cross_category_score': cross_category_score,
                    'customer_avg_basket_size': profile.get('avg_basket_size', 1),
                    'customer_purchase_frequency': profile.get('purchase_frequency', 1),
                    'month': month,
                    'day_of_week': dow,
                    'prediction_date': prediction_date
                })
        
        return pd.DataFrame(features)
    
    def create_training_data(self, df: pd.DataFrame, train_end_date: datetime) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Create training data with proper temporal split and realistic negative sampling.
        """
        # Use historical data for training
        train_data = df[df['purchase_date'] < train_end_date]
        
        # Create features for multiple time points (time series approach)
        all_features = []
        
        # Generate training examples for last 4 weeks before train_end_date
        for weeks_back in range(4):
            prediction_date = train_end_date - timedelta(weeks=weeks_back)
            week_features = self.engineer_features(train_data, prediction_date)
            
            # Add target variable (did customer buy this product in the next week?)
            target_start = prediction_date
            target_end = prediction_date + timedelta(days=7)
            
            targets = []
            for _, row in week_features.iterrows():
                purchased = df[
                    (df['customer_id'] == row['customer_id']) &
                    (df['product_id'] == row['product_id']) &
                    (df['purchase_date'] >= target_start) &
                    (df['purchase_date'] < target_end)
                ].shape[0] > 0
                targets.append(1 if purchased else 0)
            
            week_features['target'] = targets
            all_features.append(week_features)
        
        combined_features = pd.concat(all_features, ignore_index=True)
        
        # Balanced sampling (important for FMCG where purchases are sparse)
        positive_samples = combined_features[combined_features['target'] == 1]
        negative_samples = combined_features[combined_features['target'] == 0]
        
        # Sample negatives to balance classes (but not 1:1, as that's unrealistic)
        n_negatives = min(len(negative_samples), len(positive_samples) * 5)  # 5:1 ratio
        negative_samples = negative_samples.sample(n=n_negatives, random_state=42)
        
        balanced_data = pd.concat([positive_samples, negative_samples], ignore_index=True)
        
        # Prepare features and targets
        feature_cols = [col for col in balanced_data.columns if col not in ['customer_id', 'product_id', 'location_id', 'target', 'prediction_date']]
        
        X = balanced_data[feature_cols]
        y = balanced_data['target']
        
        return X, y, balanced_data
    
    def train_model(self, X: pd.DataFrame, y: pd.Series) -> None:
        """
        Train LightGBM model (better than FM for tabular data with mixed features).
        """
        # Encode categorical features
        categorical_features = ['month', 'day_of_week']
        
        # Scale numerical features
        numerical_features = [col for col in X.columns if col not in categorical_features]
        
        self.scalers['numerical'] = StandardScaler()
        X_scaled = X.copy()
        X_scaled[numerical_features] = self.scalers['numerical'].fit_transform(X[numerical_features])
        
        # Use TimeSeriesSplit for proper validation
        tscv = TimeSeriesSplit(n_splits=3)
        
        # LightGBM parameters optimized for recommendation tasks
        params = {
            'objective': 'binary',
            'metric': 'auc',
            'boosting_type': 'gbdt',
            'num_leaves': 31,
            'learning_rate': 0.1,
            'feature_fraction': 0.9,
            'bagging_fraction': 0.8,
            'bagging_freq': 5,
            'verbose': -1,
            'random_state': 42
        }
        
        # Train with cross-validation
        cv_scores = []
        for train_idx, val_idx in tscv.split(X_scaled):
            X_train_cv, X_val_cv = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
            y_train_cv, y_val_cv = y.iloc[train_idx], y.iloc[val_idx]
            
            train_data = lgb.Dataset(X_train_cv, label=y_train_cv)
            val_data = lgb.Dataset(X_val_cv, label=y_val_cv, reference=train_data)
            
            model = lgb.train(
                params,
                train_data,
                valid_sets=[val_data],
                num_boost_round=200,
                callbacks=[lgb.early_stopping(20), lgb.log_evaluation(0)]
            )
            
            y_pred = model.predict(X_val_cv, num_iteration=model.best_iteration)
            auc = roc_auc_score(y_val_cv, y_pred)
            cv_scores.append(auc)
        
        logging.info(f"Cross-validation AUC: {np.mean(cv_scores):.4f} (+/- {np.std(cv_scores)*2:.4f})")
        
        # Train final model on all data
        train_data = lgb.Dataset(X_scaled, label=y)
        self.model = lgb.train(params, train_data, num_boost_round=200)
        
        # Store feature columns for prediction
        self.feature_columns = X_scaled.columns.tolist()
        
        logging.info("Model training completed successfully")
    
    def generate_recommendations(self, customer_id: str, location_id: str, 
                               prediction_date: datetime, top_k: int = 10) -> List[Dict]:
        """
        Generate top-k recommendations for a customer at a specific date.
        """
        if not self.model:
            raise ValueError("Model not trained. Call train_model first.")
        
        # Create feature data for this customer
        # In production, this would be more efficient with pre-computed features
        mock_df = self._generate_mock_data(prediction_date - timedelta(days=90), prediction_date)
        customer_features = self.engineer_features(mock_df, prediction_date)
        customer_features = customer_features[customer_features['customer_id'] == customer_id]
        
        if len(customer_features) == 0:
            # Cold start - return popular products
            popular_products = mock_df[mock_df['location_id'] == location_id]['product_id'].value_counts().head(top_k)
            return [{'product_id': prod, 'score': 0.5, 'reason': 'popular_in_location'} 
                   for prod in popular_products.index]
        
        # Prepare features for prediction
        X_pred = customer_features[self.feature_columns]
        
        # Scale features
        numerical_features = [col for col in X_pred.columns if col not in ['month', 'day_of_week']]
        X_pred_scaled = X_pred.copy()
        X_pred_scaled[numerical_features] = self.scalers['numerical'].transform(X_pred[numerical_features])
        
        # Get predictions
        predictions = self.model.predict(X_pred_scaled, num_iteration=self.model.best_iteration)
        
        # Create recommendations with business logic
        recommendations = []
        for idx, (_, row) in enumerate(customer_features.iterrows()):
            score = predictions[idx]
            
            # Add business rules
            reason = self._get_recommendation_reason(row, score)
            
            recommendations.append({
                'product_id': row['product_id'],
                'score': score,
                'reason': reason,
                'days_since_last_purchase': row['days_since_last_purchase'],
                'repurchase_score': row['repurchase_score']
            })
        
        # Sort by score and return top-k
        recommendations.sort(key=lambda x: x['score'], reverse=True)
        return recommendations[:top_k]
    
    def _get_recommendation_reason(self, row: pd.Series, score: float) -> str:
        """Generate human-readable recommendation reasons."""
        if row['days_since_last_purchase'] <= row['avg_repurchase_interval'] * 1.1:
            return 'due_for_repurchase'
        elif row['product_popularity'] > 0.1:
            return 'popular_in_your_area'
        elif score > 0.7:
            return 'based_on_your_preferences'
        else:
            return 'discover_new_product'
    
    def evaluate_model_performance(self, test_start_date: datetime, test_end_date: datetime) -> Dict:
        """
        Proper evaluation using future data that wasn't seen during training.
        This is the correct way to evaluate recommendation systems temporally.
        """
        logging.info(f"Evaluating model performance from {test_start_date} to {test_end_date}")
        
        # Load test data (future data not seen during training)
        test_df = self.load_data_batch(test_start_date, test_end_date)
        
        if len(test_df) == 0:
            logging.warning("No test data available for evaluation")
            return {}
        
        # Update customer profiles with historical data (up to test_start_date)
        historical_df = self.load_data_batch(test_start_date - timedelta(days=90), test_start_date)
        self.customer_profiles = self.create_customer_profiles(historical_df)
        
        # Evaluate weekly performance
        weekly_metrics = []
        current_date = test_start_date
        
        while current_date < test_end_date:
            week_end = current_date + timedelta(days=7)
            
            # Generate predictions for this week
            week_features = self.engineer_features(historical_df, current_date)
            
            if len(week_features) == 0:
                current_date = week_end
                continue
            
            # Get actual purchases in the target week
            actual_purchases = test_df[
                (test_df['purchase_date'] >= current_date) & 
                (test_df['purchase_date'] < week_end)
            ]
            
            # Create targets
            targets = []
            predictions = []
            
            X_test = week_features[self.feature_columns]
            numerical_features = [col for col in X_test.columns if col not in ['month', 'day_of_week']]
            X_test_scaled = X_test.copy()
            X_test_scaled[numerical_features] = self.scalers['numerical'].transform(X_test[numerical_features])
            
            week_predictions = self.model.predict(X_test_scaled, num_iteration=self.model.best_iteration)
            
            for idx, (_, row) in enumerate(week_features.iterrows()):
                purchased = actual_purchases[
                    (actual_purchases['customer_id'] == row['customer_id']) &
                    (actual_purchases['product_id'] == row['product_id'])
                ].shape[0] > 0
                
                targets.append(1 if purchased else 0)
                predictions.append(week_predictions[idx])
            
            if len(targets) > 0 and sum(targets) > 0:  # Only evaluate if there are positive samples
                try:
                    auc = roc_auc_score(targets, predictions)
                    ap = average_precision_score(targets, predictions)
                    
                    # Calculate NDCG@10 per customer
                    customer_ndcgs = []
                    for customer_id in week_features['customer_id'].unique():
                        customer_mask = week_features['customer_id'] == customer_id
                        if sum(np.array(targets)[customer_mask]) > 0:  # Customer has positive samples
                            customer_targets = np.array(targets)[customer_mask]
                            customer_preds = np.array(predictions)[customer_mask]
                            ndcg = ndcg_score([customer_targets], [customer_preds], k=10)
                            customer_ndcgs.append(ndcg)
                    
                    avg_ndcg = np.mean(customer_ndcgs) if customer_ndcgs else 0
                    
                    weekly_metrics.append({
                        'week_start': current_date,
                        'auc': auc,
                        'average_precision': ap,
                        'ndcg_at_10': avg_ndcg,
                        'n_customers': len(week_features['customer_id'].unique()),
                        'n_purchases': sum(targets)
                    })
                    
                    logging.info(f"Week {current_date.strftime('%Y-%m-%d')}: AUC={auc:.3f}, AP={ap:.3f}, NDCG@10={avg_ndcg:.3f}")
                
                except Exception as e:
                    logging.warning(f"Evaluation failed for week {current_date}: {e}")
            
            current_date = week_end
        
        if weekly_metrics:
            overall_metrics = {
                'avg_auc': np.mean([m['auc'] for m in weekly_metrics]),
                'avg_average_precision': np.mean([m['average_precision'] for m in weekly_metrics]),
                'avg_ndcg_at_10': np.mean([m['ndcg_at_10'] for m in weekly_metrics]),
                'total_weeks_evaluated': len(weekly_metrics),
                'weekly_details': weekly_metrics
            }
            
            logging.info(f"Overall Evaluation - Avg AUC: {overall_metrics['avg_auc']:.3f}, "
                        f"Avg AP: {overall_metrics['avg_average_precision']:.3f}, "
                        f"Avg NDCG@10: {overall_metrics['avg_ndcg_at_10']:.3f}")
            
            return overall_metrics
        else:
            logging.warning("No valid weekly metrics computed")
            return {}
    
    def save_model(self, model_path: str) -> None:
        """Save the trained model and preprocessors."""
        model_artifacts = {
            'model': self.model,
            'scalers': self.scalers,
            'feature_columns': self.feature_columns,
            'customer_profiles': self.customer_profiles,
            'config': self.config
        }
        
        with open(model_path, 'wb') as f:
            pickle.dump(model_artifacts, f)
        
        logging.info(f"Model saved to {model_path}")
    
    def load_model(self, model_path: str) -> None:
        """Load a pre-trained model and preprocessors."""
        with open(model_path, 'rb') as f:
            artifacts = pickle.load(f)
        
        self.model = artifacts['model']
        self.scalers = artifacts['scalers']
        self.feature_columns = artifacts['feature_columns']
        self.customer_profiles = artifacts['customer_profiles']
        
        logging.info(f"Model loaded from {model_path}")



## Implementation

In [4]:
# # Example usage and demonstration
# if __name__ == "__main__":
#     # Configuration
#     config = {
#         'connection_string': "DRIVER={ODBC Driver 17 for SQL Server};SERVER=localhost;DATABASE=fmcg_db;UID=user;PWD=pass",
#         'model_params': {
#             'learning_rate': 0.1,
#             'max_depth': 6,
#             'n_estimators': 200
#         }
#     }
    
#     # Initialize engine
#     engine = ProductionFMCGEngine(config)
    
#     # Training phase
#     train_start = datetime(2024, 1, 1)
#     train_end = datetime(2024, 10, 1)
    
#     # Load training data
#     train_df = engine.load_data_batch(train_start, train_end)
#     engine.customer_profiles = engine.create_customer_profiles(train_df)
    
#     # Create training data with proper temporal split
#     X_train, y_train, training_data = engine.create_training_data(train_df, train_end)
    
#     # Train model
#     engine.train_model(X_train, y_train)
    
#     # Generate recommendations for a customer
#     recommendations = engine.generate_recommendations(
#         customer_id='CUST_0001',
#         location_id='LAGOS_VI',
#         prediction_date=datetime(2024, 10, 15),
#         top_k=5
#     )
    
#     print("Recommendations for CUST_0001:")
#     for i, rec in enumerate(recommendations, 1):
#         print(f"{i}. {rec['product_id']} (Score: {rec['score']:.3f}, Reason: {rec['reason']})")
    
#     # Evaluate model performance on future data
#     test_start = datetime(2024, 10, 1)
#     test_end = datetime(2024, 11, 1)
    
#     evaluation_results = engine.evaluate_model_performance(test_start, test_end)
#     print(f"\nEvaluation Results: {evaluation_results}")
    
#     # Save model for production
#     engine.save_model("production_fmcg_model.pkl")





In [5]:
# Configuration
config = {
    'connection_string': "DRIVER={ODBC Driver 17 for SQL Server};SERVER=localhost;DATABASE=fmcg_db;UID=user;PWD=pass",
    'model_params': {
        'learning_rate': 0.1,
        'max_depth': 6,
        'n_estimators': 200
    }
}

# Initialize engine
engine = ProductionFMCGEngine(config)


In [6]:
# Training phase
train_start = datetime(2024, 1, 1)
train_end = datetime(2024, 10, 1)

# Load training data
train_df = engine.load_data_batch(train_start, train_end)
engine.customer_profiles = engine.create_customer_profiles(train_df)


2025-06-15 23:50:33,326 - ERROR - Data loading failed: ('HYT00', '[HYT00] [Microsoft][ODBC Driver 17 for SQL Server]Login timeout expired (0) (SQLDriverConnect)')


In [12]:
train_df.shape

(18687, 8)

In [11]:
# Create training data with proper temporal split
X_train, y_train, training_data = engine.create_training_data(train_df, train_end)


In [14]:
# Train model
engine.train_model(X_train, y_train)

Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[1]	valid_0's auc: 1
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[1]	valid_0's auc: 1
Training until validation scores don't improve for 20 rounds


2025-06-15 23:56:14,452 - INFO - Cross-validation AUC: nan (+/- nan)


Early stopping, best iteration is:
[1]	valid_0's auc: 1


2025-06-15 23:56:15,469 - INFO - Model training completed successfully


In [19]:
# Generate recommendations for a customer
recommendations = engine.generate_recommendations(
    customer_id='CUST_0003',
    location_id='LAGOS_VI',
    prediction_date=datetime(2024, 10, 15),
    top_k=5
)

In [21]:
print("Recommendations for CUST_0001:")
for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec['product_id']} (Score: {rec['score']:.3f}, Reason: {rec['reason']})")


Recommendations for CUST_0001:
1. NOODLES_PACK (Score: 0.000, Reason: due_for_repurchase)
2. SUGAR_1KG (Score: 0.000, Reason: due_for_repurchase)
3. MILK_1L (Score: 0.000, Reason: due_for_repurchase)
4. DETERGENT_1KG (Score: 0.000, Reason: due_for_repurchase)
5. SOAP_BAR (Score: 0.000, Reason: due_for_repurchase)


In [22]:
recommendations

[{'product_id': np.str_('NOODLES_PACK'),
  'score': np.float64(1.0098173329540495e-05),
  'reason': 'due_for_repurchase',
  'days_since_last_purchase': 10,
  'repurchase_score': 1.4202898550724639},
 {'product_id': np.str_('SUGAR_1KG'),
  'score': np.float64(9.168977140049653e-06),
  'reason': 'due_for_repurchase',
  'days_since_last_purchase': 10,
  'repurchase_score': 1.4594594594594594},
 {'product_id': np.str_('MILK_1L'),
  'score': np.float64(8.078037544635482e-06),
  'reason': 'due_for_repurchase',
  'days_since_last_purchase': 5,
  'repurchase_score': 1.6835443037974684},
 {'product_id': np.str_('DETERGENT_1KG'),
  'score': np.float64(6.9883888237478205e-06),
  'reason': 'due_for_repurchase',
  'days_since_last_purchase': 10,
  'repurchase_score': 1.6774193548387095},
 {'product_id': np.str_('SOAP_BAR'),
  'score': np.float64(3.6268121592494435e-06),
  'reason': 'due_for_repurchase',
  'days_since_last_purchase': 1,
  'repurchase_score': 1.9452054794520548}]

In [18]:
# Evaluate model performance on future data
test_start = datetime(2024, 10, 1)
test_end = datetime(2024, 11, 1)
evaluation_results = engine.evaluate_model_performance(test_start, test_end)
print(f"\nEvaluation Results: {evaluation_results}")


2025-06-15 23:57:16,449 - INFO - Evaluating model performance from 2024-10-01 00:00:00 to 2024-11-01 00:00:00
2025-06-15 23:57:31,453 - ERROR - Data loading failed: ('HYT00', '[HYT00] [Microsoft][ODBC Driver 17 for SQL Server]Login timeout expired (0) (SQLDriverConnect)')
2025-06-15 23:57:47,372 - ERROR - Data loading failed: ('HYT00', '[HYT00] [Microsoft][ODBC Driver 17 for SQL Server]Login timeout expired (0) (SQLDriverConnect)')
2025-06-15 23:57:56,447 - INFO - Week 2024-10-01: AUC=0.524, AP=0.299, NDCG@10=0.684
2025-06-15 23:58:04,560 - INFO - Week 2024-10-08: AUC=0.511, AP=0.285, NDCG@10=0.678
2025-06-15 23:58:13,578 - INFO - Week 2024-10-15: AUC=0.532, AP=0.316, NDCG@10=0.703
2025-06-15 23:58:19,890 - INFO - Week 2024-10-22: AUC=0.503, AP=0.249, NDCG@10=0.689
2025-06-15 23:58:22,770 - INFO - Week 2024-10-29: AUC=0.494, AP=0.093, NDCG@10=0.556
2025-06-15 23:58:22,771 - INFO - Overall Evaluation - Avg AUC: 0.513, Avg AP: 0.248, Avg NDCG@10: 0.662



Evaluation Results: {'avg_auc': np.float64(0.5127157149454086), 'avg_average_precision': np.float64(0.24839951457198714), 'avg_ndcg_at_10': np.float64(0.6619308289953038), 'total_weeks_evaluated': 5, 'weekly_details': [{'week_start': datetime.datetime(2024, 10, 1, 0, 0), 'auc': np.float64(0.5243594370263442), 'average_precision': np.float64(0.2990862034264462), 'ndcg_at_10': np.float64(0.6835607755751981), 'n_customers': 200, 'n_purchases': 459}, {'week_start': datetime.datetime(2024, 10, 8, 0, 0), 'auc': np.float64(0.5111692274163671), 'average_precision': np.float64(0.28484080971078346), 'ndcg_at_10': np.float64(0.6775238012403685), 'n_customers': 199, 'n_purchases': 437}, {'week_start': datetime.datetime(2024, 10, 15, 0, 0), 'auc': np.float64(0.531752800045483), 'average_precision': np.float64(0.3164235706509935), 'ndcg_at_10': np.float64(0.7034403403897923), 'n_customers': 192, 'n_purchases': 429}, {'week_start': datetime.datetime(2024, 10, 22, 0, 0), 'auc': np.float64(0.502692123

In [ ]:
# Save model for production
engine.save_model("production_fmcg_model.pkl")